# Plano 2 — Otimização, orçamento e avaliação do sistema (5.5 + 6.2)

Transforma as predições walk-forward (`y_hat_rf`) em escalações válidas via PLI, simula o patrimônio de um usuário e avalia o sistema contra a escalação ótima *a posteriori*.

- **Cenário A:** rodadas 5–18 (predições de `predicoes_walkforward.csv`).
- **Cenário B:** rodadas 19–34 (predições de `predicoes_walkforward_segundo_turno.csv`, retreino a cada rodada com tudo `< t`). Começa com o estado (saldo + time) do fim da rodada 18.

A lógica fica em módulos testáveis: `models/lineup.py` (PLI + capitão), `models/budget.py` (cartoletas), `models/simulation.py` (loop), `models/evaluation.py` (Tabela 6.2) e `models/report.py` (figuras).

## 0. Imports e configuração

In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import numpy as np
import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "database" / "players.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find data/database/players.csv")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from models.budget import ORCAMENTO_INICIAL, EstadoUsuario, valoriza_time
from models.candidates import monta_candidatos
from models.evaluation import compara_turnos_6_1, erro_por_posicao, tabela_6_2
from models.lineup import MAX_CLUBE, VAGAS_POS, escalacao_otima, escalacao_sistema, pontuacao_com_capitao
from models.report import plot_desempenho
from models.simulation import STATUS_CANDIDATOS, SimulationResult, filtra_candidatos, simula_cenario

PATH_DB = PROJECT_ROOT / "data" / "database"
PATH_OUT = PROJECT_ROOT / "models" / "outputs"

MODELO_PLI = "rf"
CENARIOS: dict[str, tuple[int, int, str, str]] = {
    # cenario: (rodada_inicial, rodada_final, arquivo de predicoes, sufixo dos outputs)
    "A": (5, 18, "predicoes_walkforward.csv", ""),
    "B": (19, 34, "predicoes_walkforward_segundo_turno.csv", "_segundo_turno"),
}

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 30)
print("PROJECT_ROOT:", PROJECT_ROOT)
print(f"Orçamento inicial: {ORCAMENTO_INICIAL} | vagas: {VAGAS_POS} | máx/clube: {MAX_CLUBE}")
print(f"Modelo no PLI: {MODELO_PLI} | status candidatos: {sorted(STATUS_CANDIDATOS)}")

## 1. Candidatos: predições + preço, clube e risco

`preco_num` vem de `df_features` e é o preço de **compra antes** da rodada. `players.csv` traz o preço **depois** da rodada (usado só para valorizar o time já escalado). Candidatos = atletas com status Provável (o filtro é reaplicado na simulação).

In [ ]:
df_features = pd.read_csv(PATH_DB / "df_features.csv")
players = pd.read_csv(PATH_DB / "players.csv")

candidatos: dict[str, pd.DataFrame] = {}
for cenario, (r_ini, r_fim, pred_file, _) in CENARIOS.items():
    pred = pd.read_csv(PATH_OUT / pred_file)
    cand = monta_candidatos(pred, df_features, MODELO_PLI)
    rodadas = sorted(cand["rodada"].unique())
    assert rodadas == list(range(r_ini, r_fim + 1)), f"{cenario}: rodadas {rodadas}"
    candidatos[cenario] = cand
    print(
        f"Cenário {cenario}: {len(cand)} candidatos, rodadas {r_ini}–{r_fim}, "
        f"~{len(cand) / len(rodadas):.0f}/rodada"
    )

candidatos["A"].head(3)

## 2. PLI + capitão (5.5.1, 5.5.2, 5.5.4)

Variável binária $x_i = 1$ se o atleta $i$ é escalado:

$$\max \sum_i \hat{y}_i x_i \quad \text{s.a.} \quad \sum_i p_i x_i \le B,\quad \sum_{i \in P_k} x_i = n_k\ (k = 1..6),\quad \sum_{i \in C_j} x_i \le 5\ \forall j,\quad x_i \in \{0, 1\}$$

com $n = (1, 2, 2, 3, 3, 1)$ para goleiro, laterais, zagueiros, meias, atacantes e técnico (formação 4-3-3). Resolvido com CBC via PuLP.

- **Capitão do sistema:** entre os 11 jogadores escalados (técnico não pode ser capitão), o maior $\hat{y}_i / (\sigma_{5r,i} + \varepsilon)$ — um "Sharpe" que prefere pontuação prevista alta com baixa volatilidade recente.
- **Pontuação real:** soma dos pontos + pontos do capitão (capitão vale ×2).
- **Ótimo *a posteriori* (hindsight):** mesmo PLI com os pontos reais e com o capitão dentro do objetivo ($c_i \le x_i$, $\sum c_i = 1$), no mesmo orçamento e no mesmo conjunto de candidatos.

Exemplo na rodada 5 com o orçamento inicial:

In [ ]:
cand_r5 = filtra_candidatos(candidatos["A"], 5)

t0 = time.perf_counter()
esc_r5 = escalacao_sistema(cand_r5, ORCAMENTO_INICIAL)
tempo_pli = time.perf_counter() - t0
otima_r5 = escalacao_otima(cand_r5, ORCAMENTO_INICIAL)
assert esc_r5 is not None and otima_r5 is not None

cols_show = ["apelido", "posicao", "clube_id", "preco_num", "y_hat", "pts_std_5r", "pontos_num", "capitao"]
print(f"{len(cand_r5)} candidatos | PLI resolvido em {tempo_pli:.2f}s")
print(
    f"Custo: {esc_r5['preco_num'].sum():.2f} / {ORCAMENTO_INICIAL} | "
    f"previsto (cap x2): {pontuacao_com_capitao(esc_r5, 'y_hat'):.2f} | "
    f"real (cap x2): {pontuacao_com_capitao(esc_r5):.2f} | "
    f"ótimo: {pontuacao_com_capitao(otima_r5):.2f}"
)
esc_r5.sort_values("posicao_id")[cols_show]

## 3. Orçamento dinâmico — validação manual (rodadas 5–7)

- Rodada 5: 130 cartoletas em caixa, sem time.
- Rodada $t \ge 6$: orçamento = saldo + valor do time da rodada $t-1$ ao **preço pós-rodada** $t-1$ (= preço de compra em $t$). Assim o ganho de valorização entra no orçamento seguinte, sem usar informação da própria rodada $t$.
- Após escalar: saldo = orçamento − custo do time.

Observação: o notebook de referência valorizava o time com o preço **depois** da rodada $t$ para decidir a rodada $t$ (look-ahead); aqui isso foi corrigido.

In [ ]:
val_57 = simula_cenario(candidatos["A"], players, [5, 6, 7], EstadoUsuario(), verbose=True)
orc_57 = val_57.historico_orcamento

cols_orc = [
    "rodada_id", "orcamento_disponivel", "custo_time", "saldo_banco",
    "ganho_variacao_rodada", "patrimonio_pos_rodada", "pontos_obtidos", "pontos_otimos",
    "eficiencia_pct", "n_trocas", "capitao",
]
print(orc_57[cols_orc].round(2).to_string(index=False))

# 1) budget never exceeded; 2) wealth after t == budget of t+1; 3) conservation: wealth = saldo + team value
assert (orc_57["custo_time"] <= orc_57["orcamento_disponivel"] + 1e-6).all()
np.testing.assert_allclose(
    orc_57["orcamento_disponivel"].iloc[1:].to_numpy(),
    orc_57["patrimonio_pos_rodada"].iloc[:-1].to_numpy(),
)
np.testing.assert_allclose(
    orc_57["patrimonio_pos_rodada"].to_numpy(),
    (orc_57["orcamento_disponivel"] + orc_57["ganho_variacao_rodada"]).to_numpy(),
)

# manual check of round 5 -> 6: team bought in R5, valued at post-R5 prices
esc_57 = val_57.historico_escalacoes.query("escalacao == 'sistema'")
time_r5 = esc_57[esc_57["rodada_id"] == 5]
val_r5 = valoriza_time(time_r5["atleta_id"].tolist(), 5, players)
check = time_r5[["apelido", "preco_entrada", "variacao_rodada"]].assign(
    preco_pos_r5=time_r5["atleta_id"].map(val_r5.detalhe.set_index("atleta_id")["preco_num"])
)
np.testing.assert_allclose(check["preco_entrada"] + check["variacao_rodada"], check["preco_pos_r5"])
print(f"\nTime R5: compra {check['preco_entrada'].sum():.2f} + variação {check['variacao_rodada'].sum():+.2f} "
      f"= {check['preco_pos_r5'].sum():.2f}; saldo R5 {orc_57.loc[0, 'saldo_banco']:.2f} "
      f"-> orçamento R6 {orc_57.loc[1, 'orcamento_disponivel']:.2f}")
print("Validação R5–R7 OK")
check

## 4. Loop end-to-end (Cenário A: 5–18, Cenário B: 19–34)

Para cada rodada: orçamento → candidatos → PLI + capitão → ótimo *a posteriori* no mesmo orçamento → pontos reais com capitão ×2 → eficiência → atualiza saldo/time. O Cenário B continua do estado final do A (saldo e time da rodada 18, valorizados ao preço pós-rodada 18).

In [ ]:
resultados: dict[str, SimulationResult] = {}
patrimonio_inicial: dict[str, float] = {}
estado = EstadoUsuario()

for cenario, (r_ini, r_fim, _, tag) in CENARIOS.items():
    print(f"\n=== Cenário {cenario}: rodadas {r_ini}–{r_fim} ===")
    patrimonio_inicial[cenario] = (
        ORCAMENTO_INICIAL
        if not resultados
        else float(list(resultados.values())[-1].historico_orcamento["patrimonio_pos_rodada"].iloc[-1])
    )
    res = simula_cenario(candidatos[cenario], players, list(range(r_ini, r_fim + 1)), estado)
    resultados[cenario] = res
    estado = res.estado_final

    res.historico_orcamento.to_csv(PATH_OUT / f"historico_orcamento{tag}.csv", index=False)
    res.historico_escalacoes.to_csv(PATH_OUT / f"historico_escalacoes{tag}.csv", index=False)
    print(f"Salvos: historico_orcamento{tag}.csv, historico_escalacoes{tag}.csv")

np.testing.assert_allclose(
    resultados["B"].historico_orcamento["orcamento_disponivel"].iloc[0],
    resultados["A"].historico_orcamento["patrimonio_pos_rodada"].iloc[-1],
)
print("\nContinuidade A -> B OK (orçamento R19 = patrimônio após R18)")

## 5. Avaliação 6.2

- **Tabela 6.2** (`tabela_6_2.csv`): pontos do sistema vs. ótimo, eficiência média e agregada (Σ sistema / Σ ótimo), patrimônio inicial/final, trocas, erro de previsão dos escalados.
- **Erro por posição** (`tabela_6_2_erro_posicao.csv`): MAE e viés (real − previsto) dos atletas escolhidos pelo sistema. Viés negativo = o modelo superestimou quem escalou (efeito "maldição do vencedor" do otimizador).
- **Comparação entre turnos da Tabela 6.1** (`tabela_6_1_comparacao_turnos.csv`).

In [ ]:
tab_62 = tabela_6_2(resultados, patrimonio_inicial)
tab_erro_pos = erro_por_posicao(resultados)
tab_turnos = compara_turnos_6_1(
    pd.read_csv(PATH_OUT / "tabela_6_1.csv"),
    pd.read_csv(PATH_OUT / "tabela_6_1_segundo_turno.csv"),
)

tab_62.to_csv(PATH_OUT / "tabela_6_2.csv", index=False)
tab_erro_pos.to_csv(PATH_OUT / "tabela_6_2_erro_posicao.csv", index=False)
tab_turnos.to_csv(PATH_OUT / "tabela_6_1_comparacao_turnos.csv")

print("=== Tabela 6.2 ===")
print(tab_62.T.to_string())
print("\n=== Erro dos escalados por posição ===")
print(tab_erro_pos.to_string(index=False))
print("\n=== Tabela 6.1: primeiro vs. segundo turno ===")
print(tab_turnos[["MAE_1turno", "MAE_2turno", "delta_MAE", "Spearman_1turno", "Spearman_2turno", "delta_Spearman"]].to_string())

In [ ]:
FIGURAS = {"A": "fig_desempenho_sistema.png", "B": "fig_desempenho_segundo_turno.png"}
for cenario, (r_ini, r_fim, _, _) in CENARIOS.items():
    plot_desempenho(
        resultados[cenario].historico_orcamento,
        titulo=f"Desempenho do Sistema — Cenário {cenario} (rodadas {r_ini}–{r_fim})",
        path=PATH_OUT / FIGURAS[cenario],
        label_sistema=f"Sistema ({MODELO_PLI.upper()})",
        patrimonio_inicial=patrimonio_inicial[cenario],
    )
    print(FIGURAS[cenario], "OK")

## 6. Texto para as seções 5.5 e 6.2 (rascunho)

Números da execução de 09/11/2026 (`tabela_6_2.csv`, `tabela_6_2_erro_posicao.csv`, `tabela_6_1_comparacao_turnos.csv`). Se o notebook for re-executado com outros dados, atualizar.

### 5.5 Otimização da escalação

- **Formulação (5.5.1):** problema da mochila multiescolha (MCKP) binário: maximizar a soma das pontuações previstas sujeito ao orçamento em cartoletas, às vagas fixas por posição (1 GOL, 2 LAT, 2 ZAG, 3 MEI, 3 ATA, 1 TEC — formação 4-3-3) e a no máximo 5 atletas por clube.
- **Solução (5.5.2):** Programação Linear Inteira com CBC via PuLP. Com ~650 candidatos por rodada o ótimo exato sai em ~0,1–0,3 s, então não há ganho prático em heurísticas; o Algoritmo Genético (5.5.3) fica como alternativa metodológica não implementada.
- **Capitão (5.5.4):** após o PLI, o capitão é o jogador (exceto técnico) com maior $\hat{y}/(\sigma_{5r} + \varepsilon)$; na avaliação seus pontos contam em dobro.
- **Orçamento dinâmico:** 130 cartoletas na rodada 5; a cada rodada o orçamento é saldo + valor de mercado do time anterior após a rodada anterior. O sistema re-otimiza do zero toda rodada (sem custo de troca, como no jogo).

### 6.2 Avaliação do sistema

| | Cenário A (5–18) | Cenário B (19–34) |
|---|---|---|
| Pontos do sistema / rodada | 57,4 | 44,8 |
| Pontos ótimos (hindsight) / rodada | 186,6 | 190,6 |
| Eficiência média | 30,8% | 23,2% |
| Patrimônio | 130,0 → 97,3 (−25%) | 97,3 → 82,0 (−16%) |
| MAE / viés dos escalados | 3,87 / −1,29 | 4,02 / −2,37 |

Pontos para a discussão:

- **Eficiência contra o ótimo *a posteriori*** é uma referência inalcançável (o ótimo conhece os pontos reais e acerta os "azarões" da rodada), por isso deve ser lida junto com a referência ingênua abaixo.
- **Referência ingênua:** o mesmo PLI usando a média das últimas 5 rodadas (baseline) fez 50,4 pts/rodada no A e 45,6 no B, terminando com 69,8 cartoletas. O RF ganha ~7 pts/rodada no primeiro turno, empata no segundo e preserva mais patrimônio.
- **Maldição do vencedor:** o viés dos escalados é negativo em quase todas as posições (mais forte em atacantes: −2,1 no A, −3,5 no B). O otimizador escolhe justamente os atletas cuja previsão mais superestima o real; o previsto (~76 pts) fica bem acima do obtido.
- **Atletas que não entram em campo:** 8,9% dos escalados no A e 21,9% no B tiveram 0 ponto por não jogar, mesmo com status Provável/Dúvida. É a principal causa da queda do Cenário B.
- **Patrimônio:** cai nos dois turnos. O sistema compra atletas caros previstos acima do que pontuam, e a valorização do Cartola penaliza pontuar abaixo do esperado para o preço. Otimizar só pontos ignora a valorização.
- **Modelos no segundo turno (Tabela 6.1):** MAE ligeiramente maior (RF 1,61 → 1,65), mas Spearman maior em todos os modelos (RF 0,45 → 0,48): o ranking melhora com mais histórico e com o H2H disponível.

### Limitações e extensões

- **Cartola médio / percentil / ROI:** sem série pública da pontuação média dos usuários nem dados de ligas, a comparação fica contra o ótimo *a posteriori* e o baseline ingênuo.
- **Status pré-rodada:** o `status_id` vem do snapshot dos dados abertos e pode não refletir exatamente o status no fechamento do mercado; parte dos 22% de não-jogadores pode vir disso.
- **Extensões:** penalizar a probabilidade de não jogar no objetivo; incluir a valorização esperada (multiobjetivo pontos + patrimônio); corrigir a maldição do vencedor com shrinkage de $\hat{y}$; testar outras formações; AG como comparação metodológica.